# EP06 — NEAT Algorithm
**COMPSCI 713 · S1 2024 Q6 · 12 marks**

*Reference: Stanley & Miikkulainen (2002). Evolving neural networks through augmenting topologies.*

| Part | Topic | Marks |
|------|-------|-------|
| a | One way NEAT could be applied in a mobile robot | 6 |
| b | One difficult aspect of tuning/training for that application | 6 |

---

## Background — What is NEAT?

**NEAT (NeuroEvolution of Augmenting Topologies)** is an evolutionary algorithm that simultaneously searches for the *weights* AND *topology* (structure) of an artificial neural network.

Key innovations over simple neuroevolution:
1. **Historical markings:** each gene gets an innovation number, allowing meaningful crossover    between networks with different topologies
2. **Speciation:** new topologies are protected from immediate selection pressure by competing    within their own species — giving innovations time to optimise
3. **Minimal starting topology:** starts simple (often no hidden nodes) and adds complexity    only as needed

---

## Part a — NEAT in a Mobile Robot [6 marks]

### Application: Obstacle Avoidance / Navigation Controller

**Setup:**
The robot has a ring of proximity sensors (e.g., 8 IR/sonar sensors, one per compass octant). NEAT evolves a neural network that maps sensor readings -> motor commands (left/right wheel speeds).

**How NEAT is applied:**

1. **Genome encoding:** Each individual encodes a neural network via:
   - *Node genes:* input nodes (sensors), output nodes (motors), hidden nodes
   - *Connection genes:* weight, enabled/disabled flag, historical innovation number

2. **Fitness evaluation:** Deploy each network to control the robot in a simulation;    fitness = distance travelled without collision + bonus for reaching a goal zone

3. **Evolution loop:**
   - Select top-performing individuals within each species (elitism)
   - Crossover using historical markings to align genes from different topologies
   - Mutation: add node (split connection), add connection, perturb/reset weight

4. **Speciation:** new topology variants form new species and are shielded from extinction    until they have had time to optimise their weights

5. **Result:** the algorithm discovers the minimum sufficient network complexity for the    robot's specific environment — no hand-designed architecture needed

**Why NEAT fits this:** traditional RL requires manual architecture design; NEAT lets the task determine the network structure automatically.

---

## Part b — Most Difficult Tuning Aspect [6 marks]

### Fitness Function Design (Credit Assignment Problem)

Designing a fitness function that correctly captures the desired behaviour is extremely hard:

**Naive choices fail:**
- `fitness = distance_from_start` -> robot learns to drive straight and crash into the first wall
- `fitness = time_alive` -> robot learns to spin in circles indefinitely
- `fitness = -collisions` -> robot learns to sit still

**The real challenge — sparse rewards:**
If you only reward actually reaching the goal, almost no individual in the initial population ever reaches it by chance. The evolutionary population receives zero signal to guide improvement — this is the **sparse reward / deceptive fitness landscape** problem.

**Why this takes a lot of effort:**
- You must craft *intermediate* reward components (e.g., reward facing toward goal,   penalise proximity to walls, small reward for each new grid cell visited)
- These components must be carefully weighted to avoid unintended behaviours
- Each change requires re-running the full evolution (many hours of computation)
- There is no gradient to follow — only trial-and-error of different fitness formulations

This iterative refinement of the fitness function is where most development time is spent.

---

## Code Demo — NEAT-style Neuroevolution on a robot navigation task

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# ─── Simplified robot: 1D corridor, 4 sensors, 2 outputs ─────────────────────
def simulate(weights, steps=60):
    'Returns fitness (steps survived without hitting wall).'
    W = weights.reshape(2, 4)
    pos = 5.0  # start in middle of [0,10]
    for t in range(steps):
        sensors = np.array([max(0, pos),        # dist from left wall
                             max(0, 10-pos),     # dist from right wall
                             1.0, 1.0])          # constant bias sensors
        move = np.tanh(W @ sensors).mean()
        pos += move
        if pos <= 0 or pos >= 10: return t
    return steps  # survived all steps

# ─── Evolution ───────────────────────────────────────────────────────────────
POP, GENS, GENES = 30, 40, 8
pop = np.random.randn(POP, GENES)
best_hist, mean_hist = [], []

for gen in range(GENS):
    fit = np.array([simulate(ind) for ind in pop])
    best_hist.append(fit.max())
    mean_hist.append(fit.mean())

    # Elitism: keep top 6
    elite_idx = fit.argsort()[-6:]
    elites = pop[elite_idx]

    # Reproduce
    new_pop = list(elites)  # keep elites unchanged
    while len(new_pop) < POP:
        p1, p2 = elites[np.random.randint(6)], elites[np.random.randint(6)]
        mask  = np.random.rand(GENES) > 0.5
        child = np.where(mask, p1, p2) + np.random.randn(GENES) * 0.25
        new_pop.append(child)
    pop = np.array(new_pop)

fig, ax = plt.subplots(figsize=(8,3))
ax.plot(best_hist, label='Best fitness', linewidth=2)
ax.plot(mean_hist, label='Mean fitness', linestyle='--')
ax.set_xlabel('Generation'); ax.set_ylabel('Steps survived')
ax.set_title('NEAT-style Neuroevolution — Robot Navigation')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Initial best: {best_hist[0]}  ->  Final best: {best_hist[-1]} steps')